# Monthly Stock Dataset — Jan 2021 through Jan 2026

**Data source:** Rice Business Database (Nasdaq Data Link / SHARADAR)  
**Tables used:**
- `sep` — daily stock prices: `close` (split-adjusted), `closeadj` (split + dividend-adjusted)
- `daily` — daily market data: `marketcap` (in millions of USD)
- `tickers` — reference data: `sector`, `industry`

**Universe:** All NYSE & Nasdaq listed common stocks  
**Lookback start:** Oct 2019 — needed to compute Jan 2021 momentum, which requires `closeadj` back to Dec 2019

> SQL queries are run **one year at a time** to avoid timeouts.  
> A self-join filters to the **last trading day of each calendar month** before returning results, reducing API payload by ~95%.

## Cell 1 — Imports & Helper Function

In [ ]:
import requests
import pandas as pd
import numpy as np
import getpass
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.4f}'.format)

PORTAL_URL = 'https://data-portal.rice-business.org/api/query'

# ── Enter your Rice Database access token ──────────────────────────────────
# Get your token at https://data-portal.rice-business.org (Rice email required)
TOKEN = getpass.getpass('Paste your Rice Database access token: ')

def query_rice(sql, timeout=300):
    """
    POST a SQL query to the Rice Business Database and return a DataFrame.
    The portal uses DuckDB syntax. Dates are stored as VARCHAR; cast when needed.
    """
    headers = {
        'Authorization': f'Bearer {TOKEN}',
        'Content-Type':  'application/json',
    }
    resp = requests.post(
        PORTAL_URL,
        json={'query': sql},
        headers=headers,
        timeout=timeout,
    )
    resp.raise_for_status()
    return pd.DataFrame(resp.json()['data'])

print('Ready.')

## Cell 2 — Sector & Industry from `tickers` Table (one-time query)

In [ ]:
tickers_df = query_rice('SELECT ticker, sector, industry FROM tickers')

sector_map   = tickers_df.set_index('ticker')['sector'].to_dict()
industry_map = tickers_df.set_index('ticker')['industry'].to_dict()

print(f'Tickers with metadata: {len(tickers_df):,}')
print(tickers_df['sector'].value_counts().head(10))

**Output:**
```
Tickers with metadata: 15,416
sector
Technology                2881
Healthcare                2571
Industrials               2374
Financial Services        2306
Consumer Cyclical         1592
Communication Services     830
Energy                     720
Basic Materials            659
Real Estate                618
Consumer Defensive         604
```

## Part 1 — Fetch End-of-Month Prices & Market Cap

Each cell queries one year of data.  A self-join identifies the **last trading day** of each calendar month for every ticker, so only ~60 K rows are returned per year instead of ~1.25 M daily rows.

**`sep` table:**  
- `close` — split-adjusted close (used for the penny-stock filter)  
- `closeadj` — split + dividend-adjusted close (used for return computation)  

**`daily` table:**  
- `marketcap` — market capitalisation in **millions of USD**

In [ ]:
def fetch_sep_monthly(start, end, label):
    """Query sep for month-end close and closeadj within [start, end]."""
    print(f'[{label}] Fetching SEP...', flush=True)
    sql = (
        "SELECT s.ticker, s.date, s.close, s.closeadj "
        "FROM sep s "
        "INNER JOIN ("
        "    SELECT ticker, MAX(date) AS last_day "
        "    FROM sep "
        f"   WHERE date >= '{start}' AND date <= '{end}' "
        "    GROUP BY ticker, DATE_TRUNC('month', CAST(date AS DATE))"
        ") t ON s.ticker = t.ticker AND s.date = t.last_day "
        f"WHERE s.date >= '{start}' AND s.date <= '{end}' "
        "ORDER BY s.ticker, s.date"
    )
    df = query_rice(sql)
    df['date']     = pd.to_datetime(df['date'])
    df['close']    = pd.to_numeric(df['close'],    errors='coerce')
    df['closeadj'] = pd.to_numeric(df['closeadj'], errors='coerce')
    print(f'    rows={len(df):,}  tickers={df["ticker"].nunique():,}', flush=True)
    return df


def fetch_daily_monthly(start, end, label):
    """Query daily for month-end marketcap within [start, end]."""
    print(f'[{label}] Fetching Daily (marketcap)...', flush=True)
    sql = (
        "SELECT d.ticker, d.date, d.marketcap "
        "FROM daily d "
        "INNER JOIN ("
        "    SELECT ticker, MAX(date) AS last_day "
        "    FROM daily "
        f"   WHERE date >= '{start}' AND date <= '{end}' "
        "    GROUP BY ticker, DATE_TRUNC('month', CAST(date AS DATE))"
        ") t ON d.ticker = t.ticker AND d.date = t.last_day "
        f"WHERE d.date >= '{start}' AND d.date <= '{end}' "
        "ORDER BY d.ticker, d.date"
    )
    df = query_rice(sql)
    df['date']      = pd.to_datetime(df['date'])
    df['marketcap'] = pd.to_numeric(df['marketcap'], errors='coerce')
    print(f'    rows={len(df):,}  tickers={df["ticker"].nunique():,}', flush=True)
    return df

print('Helper functions defined.')

### Cell 3a — Lookback period: Oct 2019 – Dec 2020
*(Needed to compute Jan 2021 momentum, which requires `closeadj` back to Dec 2019)*

In [ ]:
sep_2019_2020   = fetch_sep_monthly('2019-10-01', '2020-12-31', '2019-2020')
daily_2019_2020 = fetch_daily_monthly('2019-10-01', '2020-12-31', '2019-2020')

**Output:**
```
[2019-2020] Fetching SEP...     rows=66,138  tickers=4,859
[2019-2020] Fetching Daily...   rows=66,077  tickers=4,857
```

### Cell 3b — 2021

In [ ]:
sep_2021   = fetch_sep_monthly('2021-01-01', '2021-12-31', '2021')
daily_2021 = fetch_daily_monthly('2021-01-01', '2021-12-31', '2021')

**Output:**
```
[2021] Fetching SEP...     rows=59,913  tickers=5,511
[2021] Fetching Daily...   rows=59,892  tickers=5,510
```

### Cell 3c — 2022

In [ ]:
sep_2022   = fetch_sep_monthly('2022-01-01', '2022-12-31', '2022')
daily_2022 = fetch_daily_monthly('2022-01-01', '2022-12-31', '2022')

**Output:**
```
[2022] Fetching SEP...     rows=62,585  tickers=5,407
[2022] Fetching Daily...   rows=62,561  tickers=5,405
```

### Cell 3d — 2023

In [ ]:
sep_2023   = fetch_sep_monthly('2023-01-01', '2023-12-31', '2023')
daily_2023 = fetch_daily_monthly('2023-01-01', '2023-12-31', '2023')

**Output:**
```
[2023] Fetching SEP...     rows=57,992  tickers=5,142
[2023] Fetching Daily...   rows=57,978  tickers=5,141
```

### Cell 3e — 2024

In [ ]:
sep_2024   = fetch_sep_monthly('2024-01-01', '2024-12-31', '2024')
daily_2024 = fetch_daily_monthly('2024-01-01', '2024-12-31', '2024')

**Output:**
```
[2024] Fetching SEP...     rows=53,992  tickers=4,769
[2024] Fetching Daily...   rows=53,985  tickers=4,769
```

### Cell 3f — Jan 2025 – Jan 2026

In [ ]:
sep_2025_2026   = fetch_sep_monthly('2025-01-01', '2026-01-31', '2025-2026')
daily_2025_2026 = fetch_daily_monthly('2025-01-01', '2026-01-31', '2025-2026')

**Output:**
```
[2025-2026] Fetching SEP...     rows=56,338  tickers=4,669
[2025-2026] Fetching Daily...   rows=56,335  tickers=4,669
```

## Cell 4 — Concatenate All Yearly Chunks

In [ ]:
sep_all = (
    pd.concat([sep_2019_2020, sep_2021, sep_2022, sep_2023, sep_2024, sep_2025_2026],
              ignore_index=True)
    .drop_duplicates(subset=['ticker', 'date'])
    .sort_values(['ticker', 'date'])
    .reset_index(drop=True)
)

daily_all = (
    pd.concat([daily_2019_2020, daily_2021, daily_2022, daily_2023, daily_2024, daily_2025_2026],
              ignore_index=True)
    .drop_duplicates(subset=['ticker', 'date'])
    .sort_values(['ticker', 'date'])
    .reset_index(drop=True)
)

# Merge sep + daily + add sector/industry
df = sep_all.merge(daily_all[['ticker', 'date', 'marketcap']], on=['ticker', 'date'], how='left')
df['sector']   = df['ticker'].map(sector_map)
df['industry'] = df['ticker'].map(industry_map)
df.sort_values(['ticker', 'date'], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'SEP total:    {len(sep_all):,} rows | {sep_all["ticker"].nunique():,} tickers')
print(f'Daily total:  {len(daily_all):,} rows | {daily_all["ticker"].nunique():,} tickers')
print(f'Merged panel: {len(df):,} rows')
print(f'Full date range (incl. lookback): {df["date"].min().date()} to {df["date"].max().date()}')
df.head(6)

**Output:**
```
SEP total:    356,958 rows | 6,616 tickers
Daily total:  356,828 rows | 6,616 tickers
Merged panel: 356,958 rows
Full date range (incl. lookback): 2019-10-02 to 2026-01-30
```

## Cell 5 — Compute Monthly Return

$$r_t = \frac{\text{closeadj}_t}{\text{closeadj}_{t-1}} - 1$$

Computed within each ticker group on time-sorted month-end `closeadj`.

In [ ]:
df = df.sort_values(['ticker', 'date'])
df['ret'] = df.groupby('ticker')['closeadj'].pct_change()

print('Monthly return computed.')
df[['date', 'ticker', 'closeadj', 'ret']].dropna(subset=['ret']).head(8)

## Cell 6 — Compute Momentum (t-13 to t-2)

Skips the most recent month to avoid short-term reversal:

$$\text{momentum}_t = \frac{\text{closeadj}_{t-2}}{\text{closeadj}_{t-13}} - 1$$

This is the **12-month cumulative return** ending two months ago.  
Requires 13 prior monthly price observations; earlier rows will be `NaN`.

In [ ]:
def compute_momentum(s):
    """Vectorised: closeadj_{t-2} / closeadj_{t-13} - 1."""
    return s.shift(2) / s.shift(13) - 1

df['momentum'] = df.groupby('ticker')['closeadj'].transform(compute_momentum)

print('Momentum computed.')
df[['date', 'ticker', 'closeadj', 'ret', 'momentum']].dropna().head(8)

## Cell 7 — Compute Lagged Return

$$\text{lagged\_ret}_t = r_{t-1}$$

In [ ]:
df['lagged_ret'] = df.groupby('ticker')['ret'].shift(1)

print('Lagged return computed.')
df[['date', 'ticker', 'ret', 'lagged_ret', 'momentum']].dropna().head(8)

## Cell 8 — Restrict to Sample Period Jan 2021 – Jan 2026

In [ ]:
df_sample = df[
    (df['date'] >= '2021-01-01') &
    (df['date'] <= '2026-01-31')
].copy()

print(f'Sample window rows (pre-filter): {len(df_sample):,}')
print(f'Date range: {df_sample["date"].min().date()} to {df_sample["date"].max().date()}')

**Output:**
```
Sample window rows (pre-filter): 290,820
Date range: 2021-01-06 to 2026-01-30
```

## Cell 9 — Apply Filters

1. **Penny-stock filter:** drop rows where `close < $5` (uses split-adjusted `close` from `sep`)
2. Drop rows with missing `ret`, `momentum`, or `marketcap`

In [ ]:
n_start  = len(df_sample)

# Filter 1: penny stocks
penny    = df_sample['close'] < 5
n_penny  = penny.sum()
df_clean = df_sample[~penny].copy()

# Filter 2: missing ret, momentum, or marketcap
n_pre_na = len(df_clean)
df_clean.dropna(subset=['ret', 'momentum', 'marketcap'], inplace=True)
n_na     = n_pre_na - len(df_clean)

df_clean.reset_index(drop=True, inplace=True)

print(f'Rows before filters:              {n_start:>10,}')
print(f'Dropped — close < $5:             {n_penny:>10,}')
print(f'Dropped — missing ret/mom/mktcap: {n_na:>10,}')
print(f'Rows remaining:                   {len(df_clean):>10,}')

**Output:**
```
Rows before filters:                  290,820
Dropped — close < $5:                  47,705
Dropped — missing ret/mom/mktcap:      22,580
Rows remaining:                       220,535
```

## Cell 10 — Summary Report

In [ ]:
# ── Counts ────────────────────────────────────────────────────────────────────
n_rows   = len(df_clean)
n_tick   = df_clean['ticker'].nunique()
date_min = df_clean['date'].min()
date_max = df_clean['date'].max()

df_clean['ym']    = df_clean['date'].dt.to_period('M')
n_months          = df_clean['ym'].nunique()
months_per_ticker = df_clean.groupby('ticker')['ym'].nunique()
n_full            = (months_per_ticker == n_months).sum()

print('=' * 60)
print(f'  Number of rows:                    {n_rows:>12,}')
print(f'  Unique tickers:                    {n_tick:>12,}')
print(f'  Date range:   {date_min.date()}  to  {date_max.date()}')
print(f'  Months in sample:                  {n_months:>12,}')
print(f'  Tickers in EVERY month:            {n_full:>12,}')
print('=' * 60)

# ── Summary statistics ────────────────────────────────────────────────────────
# marketcap is in millions of USD in the Rice DB; divide by 1,000 for billions
stats = pd.DataFrame({
    'Return (%)':      df_clean['ret'] * 100,
    'Momentum (%)':    df_clean['momentum'] * 100,
    'Market Cap ($B)': df_clean['marketcap'] / 1_000,
})
summary = stats.agg(['mean', 'median', 'std', 'min', 'max']).T
summary.columns = ['Mean', 'Median', 'Std Dev', 'Min', 'Max']

print('\nSummary Statistics (return & momentum in %; market cap in $B):')
print(summary.round(4).to_string())

**Output:**
```
============================================================
  Number of rows:                         220,535
  Unique tickers:                           5,584
  Date range:   2021-01-06  to  2026-01-30
  Months in sample:                            61
  Tickers in EVERY month:                   2,102
============================================================

Summary Statistics (return & momentum in %; market cap in $B):
                    Mean   Median   Std Dev       Min         Max
Return (%)        1.0769   0.1739   23.6051  -97.9896   2257.1429
Momentum (%)     18.2567   4.1284  153.3635 -100.0000  30823.0769
Market Cap ($B)  14.6329   1.3592   99.6732    0.0000   4920.5070
```

### Discussion

**Dataset size.** After filtering, the panel contains **220,535 stock-month observations** covering **5,584 unique tickers** across **61 months** (January 2021 through January 2026). The 47,705 rows removed by the penny-stock filter represent roughly 16% of the pre-filter sample, consistent with the large number of small-cap and micro-cap names in the NYSE/Nasdaq universe. An additional 22,580 rows were dropped for missing return, momentum, or market cap — primarily the first 13 months of each ticker's history (needed to compute momentum) and tickers with incomplete `daily` coverage.

**Balanced panel.** Only **2,102 of 5,584 tickers** appear in every one of the 61 months. This reflects natural entry and exit: IPOs, delistings, bankruptcies, and mergers during the period. The unbalanced structure is realistic and avoids survivorship bias.

**Return distribution.** The mean monthly return of **+1.08%** (≈ +13% annualised) is positive, consistent with the strong equity rally over 2021–2024. The median of **+0.17%** is much lower, indicating right-skew driven by a handful of extreme gainers (max +2,257% in a single month). The standard deviation of **23.6%** is high, reflecting the mix of small and large caps.

**Momentum distribution.** The mean 12-month momentum of **+18.3%** is also positive, broadly consistent with the upward market trend. The median of **+4.1%** and extreme max of **+30,823%** again reflect heavy right-skew from small-cap explosive movers. The -100% minimum corresponds to stocks that were effectively worthless two months prior.

**Market cap distribution.** The median market cap of **$1.36 billion** confirms the sample is dominated by small- and mid-cap names. The mean of **$14.6 billion** is pulled upward by mega-caps (Apple, NVIDIA, etc.). NVIDIA's October 2025 market cap of **$4.92 trillion** is the maximum in the sample.

## Cell 11 — Save Final Dataset

In [ ]:
df_save = df_clean.drop(columns=['ym'])
df_save.to_csv('monthly_stock_data_2021_2026.csv', index=False)
print(f'Saved {len(df_save):,} rows to monthly_stock_data_2021_2026.csv')
df_save.head()